# 00 · NumPy Guide for the Transformer

**Goal:** learn every NumPy tool that `numpy-transformer` uses, with special focus on **axes** and **shape changes**, so the source code stops looking like magic.

Every section ends with a line from the real code (`numpy-transformer/src/numpy_transformer/layers.py` etc.) so you can see where it is used.

Standalone — only needs `numpy`.

## Words you'll meet
- **Array (ndarray):** a rectangular grid of numbers, all of one type. Like a spreadsheet, but it can have more than two directions.
- **Shape:** how many numbers along each direction, e.g. `(32, 5, 64)` = 32 sheets, each with 5 rows and 64 columns.
- **Axis:** one direction of the array. Axis 0 is the first number in the shape, axis 1 the second, and so on. `-1` means "the last one".
- **Broadcasting:** NumPy's rule for combining arrays of different shapes by stretching the smaller one, like copying one row down a whole page.
- **View vs copy:** a view is a different *window* onto the same numbers; a copy is a new set of numbers.
- **Batched matrix multiply:** doing many small matrix multiplications at once, one per example.

## 1. Arrays, shape, dtype
Everything in the Transformer is an array. Three things describe one: its **shape**, its **number of dimensions (`ndim`)** and its **dtype** (the type of number inside).

In [ ]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])
print(a)
print("shape:", a.shape)     # (rows, columns)
print("ndim :", a.ndim)      # how many axes
print("dtype:", a.dtype)     # integers here
print("size :", a.size)      # total count of numbers

> 🧒 **In plain words:**
> `a` is a table with 2 rows and 3 columns, so its shape is `(2, 3)`. It has 2 axes (`ndim = 2`) and 6 numbers in total.
> The dtype says what kind of numbers they are. `int64` means whole numbers; token ids use this. Weights use `float64` (decimals).
> **Try it:** change one number to `1.5` and print `a.dtype` again. NumPy turns the whole array into floats, because an array has only one type.

## 2. Making arrays
The repo creates arrays with these helpers (weights start as zeros/ones/random, masks start as ones, etc.).

In [ ]:
print("zeros     :", np.zeros(3))
print("ones      :", np.ones((2, 2)))
print("full      :", np.full((2, 1), 11))              # a column of the number 11  (SOS column)
print("arange    :", np.arange(5))                      # 0,1,2,3,4  (positions)
print("zeros_like:", np.zeros_like(a))                  # zeros with the same shape/dtype as a

# dtype control
print("\nastype:", np.arange(3).astype(np.float64), " (ints turned into decimals)")
print("asarray:", np.asarray([3, 1, 2]), "(turn a list into an array; no copy if it already is one)")
print("bool   :", np.ones((2, 2), dtype=bool))

> 🧒 **In plain words:**
> These are ways to *build* an array without typing every number.
> - `np.full((n, 1), 11)` makes a column of `n` elevens: the repo uses it to put the start token `SOS` in front of every row.
> - `np.arange(max_len)` gives the positions `0, 1, 2, ...` for positional encoding.
> - `zeros_like(x)` is "give me zeros shaped like `x`", handy for gradients, which always match the shape of the thing they belong to.
> - `astype` changes the number type. `asarray` converts something to an array only if it is not one already.
> **Repo example:** `sos_col = np.full((n, 1), SOS_ID, dtype=np.int64)` in `sort_data.py`.

## 3. Random numbers
Weights start random so that different neurons learn different things. The repo uses a seeded generator so results are repeatable.

In [ ]:
rng = np.random.default_rng(42)            # seed 42 -> the same numbers every run
print(rng.normal(size=3))                  # bell-curve numbers (typical weights)
print(rng.integers(0, 10, size=5))         # random whole numbers 0..9  (sort-task digits)
print(np.random.default_rng(42).normal(size=3), " <- same seed, same numbers")

> 🧒 **In plain words:**
> A *random number generator* is a machine that makes numbers that look random but are really produced by a recipe. The **seed** (42) is the recipe's starting point: same seed, same numbers. That is why a training run can be repeated exactly.
> **Notice:** the first and last printed rows match.
> **Try it:** change 42 to 43 and see the numbers change.

## 4. Indexing and slicing
Pick out parts of an array. Square brackets take one entry per axis, separated by commas.

In [ ]:
x = np.arange(24).reshape(2, 3, 4)       # 2 sheets, 3 rows, 4 columns
print(x)
print("\nx[1, 2, 3]   =", x[1, 2, 3])        # sheet 1, row 2, column 3
print("x[0]         =\n", x[0])               # all of sheet 0
print("x[:, 1, :]   =\n", x[:, 1, :])         # row 1 from every sheet
print("x[..., -1]   =\n", x[..., -1])         # last column of everything ("..." = all the other axes)
print("x[0, :2, ::2] =\n", x[0, :2, ::2])     # first two rows, every second column

> 🧒 **In plain words:**
> Think of `x` as 2 sheets of paper, each with 3 rows and 4 columns.
> - A single number picks one entry; `:` means "all of this axis".
> - `-1` means "the last one" (counting from the end).
> - `...` (three dots) means "all the axes I did not mention". So `x[..., -1]` is "the last column, whatever the rest looks like". The repo uses `...` so the same code works for 2D, 3D and 4D arrays.
> - `a:b` is a slice (from `a` up to but not including `b`); `::2` means every second one.
> **Repo example:** `logits[0, -1, :]` in `inference.py` = "example 0, the *last* position, all vocabulary scores".

## 5. Adding an axis: `None` / `np.newaxis`
Inserting a size-1 axis is how you line shapes up for broadcasting (next section).

In [ ]:
v = np.array([10, 20, 30])
print("v shape          :", v.shape)
print("v[:, None] shape :", v[:, None].shape, "  (a column)")
print("v[None, :] shape :", v[None, :].shape, "  (a row)")
print("v[:, np.newaxis] :", v[:, np.newaxis].shape, "  (np.newaxis is just a nickname for None)")

mask_2d = np.ones((4, 5), dtype=bool)
print("\nmask[..., None, :, :] adds a 'heads' axis:", mask_2d[..., None, :, :].shape)

> 🧒 **In plain words:**
> A flat list of 3 numbers has shape `(3,)`. Writing `None` in the brackets squeezes in a new axis of length 1, turning the list into a column `(3, 1)` or a row `(1, 3)`.
> A size-1 axis is a "placeholder" that lets NumPy stretch the array later.
> **Repo examples:** `np.arange(max_len)[:, np.newaxis]` (positions as a column, in positional encoding); `mask[..., None, :, :]` (add a head axis to a mask in multi-head attention); `targets[..., None]` (in the loss).

## 6. Fancy indexing: use an array as the index
Put an **array of integers** inside the brackets and NumPy fetches those rows. This is the embedding lookup `E[ids]` from notebooks 02 and 03.

In [ ]:
E = np.arange(15).reshape(5, 3) * 10          # table: 5 rows (tokens), 3 columns (features)
ids = np.array([[0, 3],
                [3, 4]])                       # a (B=2, T=2) grid of ids
print("E:\n", E)
print("\nE[ids] shape:", E[ids].shape, "= ids.shape + (3,)")
print(E[ids])

> 🧒 **In plain words:**
> For *every number* in `ids`, NumPy goes to that row of `E` and brings it back. So the grid shape `(2, 2)` becomes `(2, 2, 3)`: each id is swapped for its row of 3 numbers. Like replacing every roll number with the student's record card.
> Notice that id `3` appears twice and gets the same row both times.
> **Repo example:** the `Embedding` layer's forward pass.

### Boolean masks
An array of `True/False` picks (or replaces) entries.

In [ ]:
s = np.array([5, -2, 7, 0])
print("s > 0       :", s > 0)
print("s[s > 0]    :", s[s > 0])                         # keep only positives
print("where       :", np.where(s > 0, s, 0))            # keep s where True, else 0
print("where -inf  :", np.where(s > 0, s, -np.inf))      # 'forbid' the False entries

> 🧒 **In plain words:**
> `s > 0` asks "is each number positive?" and returns a True/False for each. `np.where(cond, a, b)` means *for every position: if `cond` is True use `a`, otherwise use `b`*.
> Putting `-inf` (minus infinity) where we want to forbid something is the attention-mask trick: after `softmax`, `exp(-inf) = 0`, so the weight is exactly zero.
> **Repo example:** `scores = np.where(mask, scores, -np.inf)` in the attention layer. `np.where(valid_mask, targets, 0)` in the loss.

## 7. Axis operations: sum, mean, max, argmax, var
Reductions **squash** one axis into a single number. The `axis` argument says *which* direction to collapse.

Rule of thumb: **the axis you name disappears from the shape.**

In [ ]:
m = np.array([[1., 2., 3.],
              [4., 5., 6.]])        # shape (2, 3): 2 rows, 3 columns
print("m.sum()        =", m.sum(),            "  -> everything, one number")
print("m.sum(axis=0)  =", m.sum(axis=0),      "  -> collapse the rows: one total per COLUMN, shape", m.sum(axis=0).shape)
print("m.sum(axis=1)  =", m.sum(axis=1),      "  -> collapse the columns: one total per ROW, shape", m.sum(axis=1).shape)
print("m.sum(axis=-1) =", m.sum(axis=-1),     "  -> same as axis=1 here (last axis)")
print("m.mean(axis=-1)=", m.mean(axis=-1))
print("m.max(axis=-1) =", m.max(axis=-1))
print("m.argmax(axis=-1)=", m.argmax(axis=-1), "  -> the POSITION of the max in each row")

> 🧒 **In plain words:**
> Picture the grid. `axis=0` runs **down** the rows, so summing along it gives one total for each column. `axis=1` runs **across** the columns, giving one total per row.
> Easy way to remember: *the axis you name is the one that vanishes.* Shape `(2, 3)` with `axis=0` becomes `(3,)`; with `axis=1` becomes `(2,)`.
> `axis=-1` means "the last axis", so the code works whatever the number of dimensions. In the Transformer the last axis is nearly always the feature (or vocabulary) axis.
> `argmax` returns *where* the biggest number is, not the number itself; generation uses it to pick the most likely word.
> **Repo example:** `int(np.argmax(logits[0, -1, :]))` in `inference.py` chooses the next token.

### Three axes
With a `(B, T, d)` array the same rule applies:

In [ ]:
B, T, d = 2, 3, 4
y = np.arange(B * T * d, dtype=float).reshape(B, T, d)
print("y shape          :", y.shape)
print("sum(axis=0)      :", y.sum(axis=0).shape, " (add up the examples)")
print("sum(axis=1)      :", y.sum(axis=1).shape, " (add up the positions)")
print("sum(axis=-1)     :", y.sum(axis=-1).shape, " (add up the features of each token)")
print("sum(axis=(0, 1)) :", y.sum(axis=(0, 1)).shape, " (collapse examples AND positions -> one number per feature)")

> 🧒 **In plain words:**
> Whichever axis you name vanishes. You can also name **several at once** with a tuple like `(0, 1)`, and all of those vanish.
> **Repo example:** in `LayerNorm.backward`, `sum_axes = tuple(range(grad_output.ndim - 1))` means "every axis except the last". The bias gradient must have shape `(d,)`, so all the other axes are added up.
> **Try it:** predict the shape of `y.sum(axis=(1, 2))` before running it.

## 8. `keepdims=True`: keep the squashed axis (as length 1)
Normally the collapsed axis disappears. With `keepdims=True` it stays as a size-1 axis, so the result can be **broadcast back** against the original.

In [ ]:
z = np.array([[1., 2., 3.],
              [4., 5., 6.]])
mx_plain = z.max(axis=-1)                     # shape (2,)
mx_keep  = z.max(axis=-1, keepdims=True)      # shape (2, 1)
print("without keepdims:", mx_plain.shape, mx_plain)
print("with    keepdims:", mx_keep.shape)
print(mx_keep)

print("\nz - mx_keep (works, row-wise):\n", z - mx_keep)
try:
    print(z - mx_plain)
except ValueError as e:
    print("\nz - mx_plain fails:", e)

> 🧒 **In plain words:**
> The max of each row has two forms: a flat list `[3, 6]` (shape `(2,)`) or a column `[[3], [6]]` (shape `(2, 1)`). Only the **column** can be subtracted from each row: NumPy lines up the `(2, 1)` against `(2, 3)` and stretches the column across.
> The flat version gets lined up with the *columns* instead and, here, the shapes don't match, so it fails.
> **Rule:** use `keepdims=True` whenever you plan to use the result against the original array.
> **Repo example (the softmax):** `x - x.max(axis=-1, keepdims=True)`, `exp_x / exp_x.sum(axis=-1, keepdims=True)`.

## 9. Broadcasting
How NumPy combines arrays of **different shapes**. Compare the shapes **from the right**; at each position the two sizes must be **equal** or **one of them must be 1** (size 1 is stretched). Missing axes on the left count as 1.

In [ ]:
def can_broadcast(s1, s2):
    # returns the result shape or None
    out = []
    for a_, b_ in zip(reversed(s1), reversed(s2)):
        if a_ == b_ or a_ == 1 or b_ == 1:
            out.append(max(a_, b_))
        else:
            return None
    longer = s1 if len(s1) > len(s2) else s2
    out += list(reversed(longer))[len(out):]
    return tuple(reversed(out))

cases = [
    ((32, 5, 64), (64,)),        # bias added to every token       (Linear: x @ W + b)
    ((32, 5, 64), (5, 64)),      # positional encoding added to every example
    ((2, 3), (2, 1)),            # row-wise max subtracted
    ((2, 3), (2,)),              # does NOT line up
    ((8, 4, 5, 5), (5, 5)),      # a (T,T) mask applied to (B, h, T, T) scores
]
for s1, s2 in cases:
    print(f"{str(s1):16} with {str(s2):10} -> {can_broadcast(s1, s2)}")

> 🧒 **In plain words:**
> Imagine writing the two shapes on top of each other, **right-aligned**:
> ```
> (32, 5, 64)
>       (64,)    -> stretched copies: the 64 numbers are added to every token
> ```
> At each column of the picture the two numbers must match, or one must be `1` (then it is stretched to fit), or be missing (also stretched).
> This is how a single bias vector `b` of length 64 is added to all `32 × 5` tokens without any loop. It is also how one positional-encoding table `(T, d)` is added to every example in the batch.
> **Notice:** `(2, 3)` with `(2,)` fails, but `(2, 3)` with `(2, 1)` works. That is exactly why `keepdims=True` exists.

In [ ]:
# prove it with real arrays
x = np.ones((2, 5, 4))
b = np.array([1., 2., 3., 4.])
print((x + b).shape, "   x + b[0] =", (x + b)[0, 0])

pe = np.arange(5 * 4).reshape(5, 4)
print((x + pe).shape, "  every example gets the same positional table")

> 🧒 **In plain words:**
> Same idea with actual numbers: adding a length-4 list `b` to a `(2, 5, 4)` block adds `[1, 2, 3, 4]` to *each* of the 10 token vectors. The shape is unchanged.

## 10. `reshape`: same numbers, new grid
`reshape` re-labels the same numbers with a different shape. The total count must stay the same. A single **`-1`** means "work this length out for me".

In [ ]:
r = np.arange(12)
print("r             :", r)
print("reshape(3, 4) :\n", r.reshape(3, 4))
print("reshape(2,2,3) shape:", r.reshape(2, 2, 3).shape)
print("reshape(-1, 6) shape:", r.reshape(-1, 6).shape, " (-1 became 2, since 12 / 6 = 2)")
print("reshape(-1)   shape:", r.reshape(-1).shape, " (flatten into a list)")

> 🧒 **In plain words:**
> Think of 12 cards in a line. `reshape(3, 4)` deals them into 3 rows of 4, in order, **left to right then top to bottom**. The cards themselves don't change; only how you stack them.
> A `-1` means "figure out this number so that everything fits". It is a handy shortcut so you don't have to compute it yourself.
> **Try it:** what happens with `r.reshape(5, 3)`? (A `ValueError`: 12 cards can't make a 5 × 3 grid.)

### The "flatten batch and sequence" trick (used in `Linear.backward`)
The weight gradient formula `xᵀ @ grad` only works for a **2-D** matrix, but our data is 3-D `(B, T, features)`. Fix: merge the first two axes with `reshape(-1, features)`.

In [ ]:
B, T, d_in, d_out = 32, 5, 32, 64
x = np.random.default_rng(0).normal(size=(B, T, d_in))
grad = np.random.default_rng(1).normal(size=(B, T, d_out))

x_2d = x.reshape(-1, x.shape[-1])               # (160, 32): 160 token vectors
grad_2d = grad.reshape(-1, grad.shape[-1])      # (160, 64)
print("x_2d", x_2d.shape, " grad_2d", grad_2d.shape)

grad_W = x_2d.T @ grad_2d                       # (32,160) @ (160,64) -> (32,64)
grad_b = grad_2d.sum(axis=0)                    # add up the 160 rows -> (64,)
print("grad_W", grad_W.shape, "  grad_b", grad_b.shape)

> 🧒 **In plain words:**
> The model sees 32 sentences of 5 tokens, but for the weight update every token is just one more training example. So we pile all `32 × 5 = 160` token vectors into one tall list `(160, 32)`, and then an ordinary 2-D matrix multiply collects the contributions of all 160 at once.
> `x_2d.T` flips rows and columns, so `(160, 32)` becomes `(32, 160)`. Multiply by `(160, 64)` and the inner `160`s cancel, leaving `(32, 64)`: **exactly the shape of the weight `W`**. A gradient always has the same shape as the thing it updates, which is a very handy sanity check.

## 11. `swapaxes` and `.T`: swapping directions
`reshape` re-deals the cards; **`swapaxes` actually swaps two directions** (the data is *read* in a different order).

In [ ]:
q = np.arange(6).reshape(2, 3)
print("q:\n", q)
print("q.T:\n", q.T)                       # swap the only two axes
print("swapaxes(q, 0, 1) same as .T:", np.array_equal(np.swapaxes(q, 0, 1), q.T))

t = np.arange(24).reshape(2, 3, 4)          # (B, T, d)
print("\nt.shape              :", t.shape)
print("swapaxes(t, -1, -2)  :", np.swapaxes(t, -1, -2).shape, " <- transposes each sheet separately")
print("t.T would give       :", t.T.shape, " <- reverses ALL axes (usually not what we want)")

> 🧒 **In plain words:**
> `.T` flips a table over its diagonal: rows become columns. For tables with more than 2 axes, `.T` reverses **all** the axes, which is rarely what you want.
> `np.swapaxes(t, -1, -2)` swaps just the **last two** axes, i.e. it transposes every `(T, d)` sheet inside the batch and leaves the batch axis alone. This is what attention needs for `Kᵀ`.
> **Repo example:** `scores = np.matmul(Q, np.swapaxes(K, -1, -2)) * scale`.

## 12. Matrix multiplication: `@` and `np.matmul`
`A @ B` multiplies matrices: `(m, n) @ (n, p) → (m, p)` — the **inner numbers must match** and they cancel. With extra leading axes, it multiplies each pair of matrices **independently** (batched).

In [ ]:
A = np.ones((2, 3))
Bm = np.ones((3, 4))
print("(2,3) @ (3,4) ->", (A @ Bm).shape)

# batched: leading axes are carried along
Q = np.random.default_rng(0).normal(size=(2, 4, 5, 8))   # (B, heads, T, d_k)
K = np.random.default_rng(1).normal(size=(2, 4, 5, 8))
scores = np.matmul(Q, np.swapaxes(K, -1, -2))            # (..,5,8) @ (..,8,5) -> (..,5,5)
print("Q", Q.shape, " K^T", np.swapaxes(K, -1, -2).shape, " -> scores", scores.shape)

# a matrix against a 3-D batch: the matrix is reused for every example
x = np.ones((32, 5, 32)); W = np.ones((32, 64))
print("(32,5,32) @ (32,64) ->", (x @ W).shape)
print("@ and np.matmul are the same:", np.allclose(x @ W, np.matmul(x, W)))

> 🧒 **In plain words:**
> Only the **last two axes** do the matrix multiplication; everything before them is a "batch of matrices" and is just carried along. So `(2, 4, 5, 8) @ (2, 4, 8, 5)` does `2 × 4 = 8` separate `(5×8) @ (8×5)` multiplications in one call and gives `(2, 4, 5, 5)`: a `5 × 5` attention-score table for every example and every head.
> When one side has fewer axes (like the weight `(32, 64)`), NumPy reuses it for every example: same layer weights for the whole batch.
> **Shape check habit:** write the shapes, cancel the inner pair, and what remains is the result. `(5×8)·(8×5) → (5×5)`.

## 13. Splitting into heads: `reshape` + `swapaxes`
The multi-head attention trick, using only what we've learned. We want `(B, T, d_model)` → `(B, heads, T, d_k)`.

In [ ]:
B, T, d_model, h = 2, 3, 8, 2
d_k = d_model // h
x = np.arange(B * T * d_model).reshape(B, T, d_model)
print("x                 :", x.shape)

step1 = x.reshape(B, T, h, d_k)            # cut the last axis (8) into 2 heads of 4
print("1. reshape        :", step1.shape)

step2 = np.swapaxes(step1, -2, -3)         # bring the 'heads' axis forward
print("2. swapaxes(-2,-3):", step2.shape)

# which numbers does head 0 own?
print("\ntoken 0 of example 0, full vector :", x[0, 0])
print("head 0 sees                         :", step2[0, 0, 0])
print("head 1 sees                         :", step2[0, 1, 0])

# merge back (inverse)
merged = np.swapaxes(step2, -2, -3).reshape(B, T, d_model)
print("\nround trip is exact:", np.array_equal(merged, x))

> 🧒 **In plain words:**
> Each token is a vector of 8 numbers. To give each of the 2 heads its own half:
> 1. `reshape(B, T, 2, 4)` cuts the 8 numbers into **2 groups of 4**. (Nothing moves; we just re-label the axis.)
> 2. `swapaxes` brings the "head" axis in front of the "token" axis, so each head holds a full `(T, d_k)` table that attention can work on separately.
> Head 0 gets the first 4 numbers (`0 1 2 3`), head 1 the last 4 (`4 5 6 7`). Merging is the exact reverse: swap back, then `reshape` to join the groups.
> **Repo example:** `_split_heads` and `_merge_heads` in `MultiHeadAttention`.

## 14. Views, copies, and contiguous memory
`reshape` and `swapaxes` usually return a **view**: a new window on the *same* numbers (fast, no copying). Changing the view changes the original!

In [ ]:
base = np.arange(6)
view = base.reshape(2, 3)
view[0, 0] = 99
print("base after changing the view:", base, " <- original changed too")

safe = base.copy()                 # an independent copy
safe[0] = -1
print("base after changing the copy:", base, " <- unchanged")

sw = np.swapaxes(np.arange(6).reshape(2, 3), 0, 1)
print("\nswapaxes result is C-contiguous?", sw.flags["C_CONTIGUOUS"], " (memory order no longer matches the shape)")
print("np.ascontiguousarray ->", np.ascontiguousarray(sw).flags["C_CONTIGUOUS"])

> 🧒 **In plain words:**
> A **view** is like looking at the same notebook through a differently shaped window: if you write on it, the notebook changes. A **copy** (`.copy()`) is a photocopy: you can scribble on it freely.
> The repo uses `.copy()` where it must not accidentally modify a saved value (e.g. when caching values for the backward pass).
> After `swapaxes` the numbers are no longer laid out in the order the new shape implies ("non-contiguous"). NumPy copes by making a copy automatically when needed (e.g. in the `reshape` after swapaxes when merging heads). `np.ascontiguousarray` forces a tidy copy if you need one.

## 15. Element-wise math
These apply to **every number** independently and keep the shape.

In [ ]:
v = np.array([0., 1., 2.])
print("exp   :", np.exp(v).round(3))
print("log   :", np.log(np.array([1., 2.718281828, 10.])).round(3))
print("sqrt  :", np.sqrt(np.array([1., 4., 9.])))
print("power :", np.power(10000.0, np.array([0., 0.5, 1.])))
print("sin/cos:", np.sin(np.pi / 2).round(3), np.cos(0.0))
print("inf   :", np.exp(-np.inf), "  <- exp of minus infinity is 0")
print("tril  :\n", np.tril(np.ones((3, 3), dtype=bool)).astype(int))

> 🧒 **In plain words:**
> - `exp`, `log`, `sqrt`, `sin`, `cos` and `power` act on each number separately.
> - `exp(-inf) = 0`: this is why a `-inf` score becomes a weight of exactly zero after softmax.
> - `np.tril` ("triangle, lower") keeps the lower-left triangle and zeros the rest. As a True/False grid it says "position `i` may look at position `j` only if `j ≤ i`": the **causal mask**.
> **Repo examples:** `np.power(10000.0, ...)`, `np.sin`, `np.cos` build positional encoding. `np.tril(np.ones((n_q, n_q), dtype=bool))` builds the decoder mask.

## 16. Softmax from scratch (axis + keepdims + broadcasting together)
The repo's softmax is four lines, and now you know every one of them.

In [ ]:
def softmax(x):
    shifted = x - x.max(axis=-1, keepdims=True)          # subtract each row's max  (broadcast)
    exp_x = np.exp(shifted)                              # element-wise
    return exp_x / exp_x.sum(axis=-1, keepdims=True)     # divide by each row's total

scores = np.array([[1., 2., 3.],
                   [1000., 1001., 1002.]])               # big numbers would overflow without the shift
p = softmax(scores)
print(p.round(3))
print("row sums:", p.sum(axis=-1))

> 🧒 **In plain words:**
> 1. **Subtract the row max** so the biggest number becomes 0. It doesn't change the answer, but stops `exp(1002)` from exploding to infinity. Both rows give the same result because they differ by a constant.
> 2. `exp` makes every number positive.
> 3. **Divide by the row total** so the row adds up to 1 (like percentages).
> All three use `axis=-1, keepdims=True` so each row is handled on its own and the `(rows, 1)` result stretches back across the row.

## 17. LayerNorm forward, line by line
```python
mean = x.mean(axis=-1, keepdims=True)
var  = x.var(axis=-1, keepdims=True)
x_norm = (x - mean) / np.sqrt(var + eps)
out = gamma * x_norm + beta
```

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(5, 3, size=(2, 3, 4))          # (B, T, d) badly scaled
gamma, beta, eps = np.ones(4), np.zeros(4), 1e-5

mean = x.mean(axis=-1, keepdims=True)         # (2,3,1): one mean per token
var  = x.var(axis=-1, keepdims=True)          # (2,3,1)
x_norm = (x - mean) / np.sqrt(var + eps)      # (2,3,4)
out = gamma * x_norm + beta                   # gamma, beta: (4,) broadcast over every token

print("mean shape:", mean.shape, "  out shape:", out.shape)
print("per-token mean after:", out.mean(axis=-1).round(6).ravel())
print("per-token std  after:", out.std(axis=-1).round(3).ravel())

> 🧒 **In plain words:**
> Every token vector (4 numbers) is rescaled on its own so its average is 0 and its spread is 1: "grading on a curve", one student (token) at a time.
> Because `axis=-1` collapses only the feature axis, the result `(2, 3, 1)` keeps one mean per token. `keepdims` lets that column stretch back across the 4 features.
> `gamma` and `beta` are length-4 lists, broadcast across all tokens.

## 18. `np.concatenate`, `np.sort`, `np.full` — building the sort-task data
```
encoder input:   unsorted digits
decoder input:   SOS + sorted digits
decoder target:  sorted digits + EOS
```

In [ ]:
SOS, EOS = 11, 12
enc = np.array([[3, 1, 4, 1, 5],
                [9, 2, 6, 5, 3]])
sorted_digits = np.sort(enc, axis=-1)                 # sort inside each row

n = enc.shape[0]
sos_col = np.full((n, 1), SOS)                        # (2,1) column of 11s
eos_col = np.full((n, 1), EOS)
dec_in  = np.concatenate([sos_col, sorted_digits], axis=1)    # glue columns together
dec_tgt = np.concatenate([sorted_digits, eos_col], axis=1)
print(dec_in); print(dec_tgt)
print("shapes:", enc.shape, dec_in.shape, dec_tgt.shape)

> 🧒 **In plain words:**
> - `np.sort(enc, axis=-1)` sorts **along the last axis**, so each row is sorted separately, not the whole table.
> - `np.concatenate([a, b], axis=1)` sticks arrays together **along axis 1 (columns)**: the shape grows along that axis only (`5` columns + `1` = `6`). All other axes must match, which is why the `SOS` column has to be `(n, 1)`.
> - With `axis=0` you would stack them on top of each other instead.
> **Repo code:** this is almost exactly `sort_data.py` lines 55-61.

## 19. Two special indexing tools used in training
### `np.put_along_axis` — set one entry per row (label smoothing in the loss)

In [ ]:
V = 5
eps = 0.1
targets = np.array([[2, 0],
                    [4, 1]])                        # (B=2, T=2): the correct token ids
p = np.full((2, 2, V), eps / (V - 1))               # (2,2,5): small probability everywhere
np.put_along_axis(p, targets[..., None], 1.0 - eps, axis=-1)   # big probability at the right id

print(p[0, 0].round(3), " <- target id 2 gets 0.9")
print("each row sums to:", p.sum(-1)[0, 0].round(3))

> 🧒 **In plain words:**
> We build the "ideal answer" distribution for each position: almost all probability (0.9) on the correct word and a tiny share (0.025) on all the others, instead of a hard 1 and 0s (that is **label smoothing**).
> `targets[..., None]` turns `(2, 2)` into `(2, 2, 1)`: one id per position. `put_along_axis(..., axis=-1)` then writes 0.9 into **that column** of each row. It works like "for each row, go to column `target` and write this value".
> **Repo example:** `training.py` lines 135-136.

### `np.add.at` — add into the same row more than once (embedding gradient)

In [ ]:
grad_weight = np.zeros((4, 2))
token_ids = np.array([1, 3, 1])                       # token 1 appears TWICE
grad_out = np.array([[1., 1.], [10., 10.], [100., 100.]])

wrong = np.zeros((4, 2))
wrong[token_ids] += grad_out                           # looks right, but isn't
print("plain +=  :\n", wrong)

np.add.at(grad_weight, token_ids, grad_out)
print("np.add.at :\n", grad_weight)

> 🧒 **In plain words:**
> Token 1 is used twice, so row 1 of the gradient should receive **both** contributions: `1 + 100 = 101`.
> A plain `wrong[token_ids] += ...` only keeps the *last* write for a repeated index, giving `100` and silently losing the `1`. `np.add.at` really **adds every contribution**, even for repeated ids.
> It matters because common words (like "the") appear many times in a batch. **Repo example:** `np.add.at(self.grad_weight, token_ids, grad_output)` in the embedding layer.

## 20. Saving and loading arrays
The repo stores trained weights in a compressed `.npz` file (a zip of named arrays).

In [ ]:
import tempfile, os
path = os.path.join(tempfile.mkdtemp(), "demo.npz")

np.savez_compressed(path, W=np.ones((2, 3)), b=np.zeros(3), step=np.array(100))
data = np.load(path)
print("keys:", data.files)
print("W:", data["W"].shape, " b:", data["b"].shape, " step:", int(data["step"]))
print("is integer dtype?", np.issubdtype(data["step"].dtype, np.integer))
print("is float dtype? ", np.issubdtype(data["W"].dtype, np.floating))

> 🧒 **In plain words:**
> `savez_compressed` packs several named arrays into one file, like a zip folder; `np.load` opens it and you read each by name (`data["W"]`).
> `np.issubdtype` asks "is this number type a kind of integer / decimal?", used when checking a loaded file before trusting it.
> **Repo:** `persistence.py` does this to save and restore a whole trained model.

## 21. Cheat sheet: where each operation appears

| Operation | What it does | Typical use in the repo |
|---|---|---|
| `x.shape`, `x.ndim`, `x.dtype` | describe an array | shape checks everywhere |
| `np.zeros/ones/full/arange` | build arrays | init, masks, `SOS` columns, positions |
| `np.zeros_like` | zeros shaped like another | gradient buffers |
| `E[ids]` (fancy indexing) | fetch rows | embedding lookup |
| `x[..., -1]`, `x[:, None]` | slice / add axis | last position, masks, positions |
| `.sum/.mean/.var/.max(axis=…)` | collapse an axis | softmax, LayerNorm, bias gradient |
| `keepdims=True` | keep collapsed axis as size 1 | broadcast back to the original |
| `np.argmax(axis=…)` | index of the largest | choose next token |
| broadcasting | stretch size-1/missing axes | bias, positional encoding, masks |
| `.reshape(…, -1)` | re-label the grid | flatten batch+seq; split/merge heads |
| `np.swapaxes(x, a, b)`, `.T` | swap directions | `Kᵀ`, heads axis |
| `@`, `np.matmul` | (batched) matrix product | every Linear layer, attention |
| `np.where(c, a, b)` | choose per element | attention and loss masks (`-inf`) |
| `np.tril` | lower triangle | causal mask |
| `np.exp/log/sqrt/sin/cos/power` | element-wise math | softmax, loss, LayerNorm, positional encoding |
| `np.concatenate(axis=…)` | glue arrays | `SOS`/`EOS` columns, growing the decoder input |
| `np.sort(axis=…)` | sort along an axis | sort-task labels |
| `np.put_along_axis` | write one entry per row | label-smoothed targets |
| `np.add.at` | add with repeated indices | embedding gradient |
| `.copy()`, `np.ascontiguousarray` | independent / tidy copy | caches, after `swapaxes` |
| `np.random.default_rng` | seeded random numbers | init, data generation |
| `np.savez_compressed`, `np.load` | save / load arrays | model persistence |

## 22. Quiz yourself (answers in the next cell)
1. `x` has shape `(32, 5, 64)`. What is the shape of `x.sum(axis=1)`?
2. And of `x.mean(axis=-1, keepdims=True)`?
3. `a` has shape `(8, 4, 5, 5)` and `m` has shape `(5, 5)`. Can you compute `a + m`? What is the shape?
4. What does `np.matmul` do with shapes `(2, 4, 5, 8)` and `(2, 4, 8, 5)`?
5. `x.reshape(-1, 64)` for the `x` in question 1: what is the first number?

In [ ]:
x = np.zeros((32, 5, 64))
print("1:", x.sum(axis=1).shape)                       # (32, 64)
print("2:", x.mean(axis=-1, keepdims=True).shape)      # (32, 5, 1)
print("3:", (np.zeros((8, 4, 5, 5)) + np.zeros((5, 5))).shape)   # (8, 4, 5, 5)
print("4:", np.matmul(np.zeros((2, 4, 5, 8)), np.zeros((2, 4, 8, 5))).shape)  # (2, 4, 5, 5)
print("5:", x.reshape(-1, 64).shape)                   # (160, 64)  because 32*5 = 160

> 🧒 **In plain words:**
> If you got these right, you can read most of `layers.py` already. Remember the two golden rules: **the axis you name disappears** (unless `keepdims=True`), and **broadcasting lines shapes up from the right**.

## Where next
➡ Open `01_vocabulary_and_tokens` to start the Transformer journey, and come back to this notebook whenever a NumPy line looks mysterious.